# Text Preprocessing and Tokenization

* * *

<div class="alert alert-success">  
    
### Learning Objectives
    
* Learn common steps for preprocessing text data, as well as specific operations for preprocessing Twitter data.
* Know commonly used NLP packages and what they are capable of.
* Understand tokenizers, and how they have changed since the advent of Large Language Models.
</div>

This is tutorial is adapted from:
- https://github.com/dlab-berkeley/Python-NLP-Fundamentals


In [3]:
# Uncomment the following lines to install packages/model
!pip install NLTK
!pip install transformers
!pip install spaCy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


<a id='section1'></a>

# Preprocessing

In Part 1 of this tutorial, we'll address the first step of text analysis. Our goal is to convert the raw, messy text data into a consistent format. This process is often called **preprocessing**, **text cleaning**, or **text normalization**.

You'll notice that at the end of preprocessing, our data is still in a format that we can read and understand. In Parts 2, we will begin our foray into converting the text data into a numerical representation—a format that can be more readily handled by computers.

🔔 **Question**: Let's pause for a minute to reflect on **your** previous experiences working on text data.
- What is the format of the text data you have interacted with (plain text, CSV, or XML)?
- Where does it come from (structured corpus, scraped from the web, survey data)?
- Is it messy (i.e., is the data formatted consistently)?


## Common Processes

Preprocessing is not something we can accomplish with a single line of code. We often start by familiarizing ourselves with the data, and along the way, we gain a clearer understanding of the granularity of preprocessing we want to apply.

Typically, we begin by applying a set of commonly used processes to clean the data. These operations don't substantially alter the form or meaning of the data; they serve as a standardized procedure to reshape the data into a consistent format.

The following processes, for examples, are commonly applied to preprocess English texts of various genres. These operations can be done using built-in Python functions, such as `string` methods, and Regular Expressions.
- Lowercase the text
- Remove punctuation marks
- Remove extra whitespace characters
- Remove stop words

After the initial processing, we may choose to perform task-specific processes, the specifics of which often depend on the downstream task we want to perform and the nature of the text data (i.e., its stylistic and linguistic features).  

Before we jump into these operations, let's take a look at our data!

### Import the Text Data

The text data we'll be working with is a CSV file. It contains tweets about U.S. airlines, scrapped from Feb 2015.

Let's read the file `airline_tweets.csv` into dataframe with `pandas`.

In [4]:
import os
import gdown
import pandas as pd

# Upload the data file to your Google Drive, and turn on the link sharing
# Replace 'YOUR_FILE_ID' with the actual file ID from your public Google Drive link e.g., "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"
file_id = '1qqIG8_-Y4JwbBj86aR_1hnaIJOkOjgxS'

# Name for the downloaded file in Colab
output_filename = 'airline_tweets.csv'

if file_id and not os.path.exists(output_filename):
    gdown.download(id=file_id, output=output_filename, quiet=False)
    print(f"File '{output_filename}' downloaded successfully!")
else:
    print(f"Using existing file '{output_filename}'.")

# Now you can load it with pandas
tweets= pd.read_csv(output_filename)
display(tweets.head())


Downloading...
From: https://drive.google.com/uc?id=1qqIG8_-Y4JwbBj86aR_1hnaIJOkOjgxS
To: /content/airline_tweets.csv
100%|██████████| 3.42M/3.42M [00:00<00:00, 46.6MB/s]


File 'airline_tweets.csv' downloaded successfully!


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


The dataframe has one row per tweet. The text of tweet is shown in the `text` column.
- `text` (`str`): the text of the tweet.

Other metadata we are interested in include:
- `airline_sentiment` (`str`): the sentiment of the tweet, labeled as "neutral," "positive," or "negative."
- `airline` (`str`): the airline that is tweeted about.
- `retweet count` (`int`): how many times the tweet was retweeted.

Let's take a look at some of the tweets:

In [5]:
print(tweets['text'].iloc[0])
print(tweets['text'].iloc[1])
print(tweets['text'].iloc[2])

@VirginAmerica What @dhepburn said.
@VirginAmerica plus you've added commercials to the experience... tacky.
@VirginAmerica I didn't today... Must mean I need to take another trip!


### Lowercasing

While we acknowledge that a word's casing is informative, we often don't work in contexts where we can properly utilize this information.

More often, the subsequent analysis we perform is **case-insensitive**. For instance, in frequency analysis, we want to account for various forms of the same word. Lowercasing the text data aids in this process and simplifies our analysis.

We can easily achieve lowercasing with the string method [`.lower()`](https://docs.python.org/3/library/stdtypes.html#str.lower); see [documentation](https://docs.python.org/3/library/stdtypes.html#string-methods) for more useful functions.

Let's apply it to the following example:

In [6]:
# Print the first example
first_example = tweets['text'][108]
print(first_example)

@VirginAmerica I was scheduled for SFO 2 DAL flight 714 today. Changed to 24th due weather. Looks like flight still on?


In [7]:
# Check if all characters are in lowercase
print("All characters are lowercase?:", first_example.islower()) # islower = f or t

# Convert it to lowercase
print("Lowercase:", first_example.lower())

# Convert it to uppercase
print("Uppercase:", first_example.upper())

All characters are lowercase?: False
Lowercase: @virginamerica i was scheduled for sfo 2 dal flight 714 today. changed to 24th due weather. looks like flight still on?
Uppercase: @VIRGINAMERICA I WAS SCHEDULED FOR SFO 2 DAL FLIGHT 714 TODAY. CHANGED TO 24TH DUE WEATHER. LOOKS LIKE FLIGHT STILL ON?


### Remove Extra Whitespace(공백) Characters

Sometimes we might come across texts with extraneous whitespace, such as spaces, tabs, and newline characters, which is particularly common when the text is scrapped from web pages. Before we dive into the details, let's briefly introduce Regular Expressions (regex) and the `re` package.

### What is a Regular Expression?

Regular expressions (regex) are patterns used to find or match text. Think of them as a "search formula" — instead of looking for one exact word, you describe what kind of text you want.

- Plain search: "find John Smith" → finds only that exact name
- Regex search: "find anything that looks like a phone number" → finds `010-1234-5678`, `02-987-6543`, etc.

Many NLP packages heavily rely on regex under the hood. Regex testers, such as [regex101](https://regex101.com), are useful tools in both understanding and creating regex expressions.

### Regex Basics  정규 표현식(regex)은 텍스트를 찾거나 일치시키는 데 사용되는 패턴입니다. 이를 '검색 공식'이라고 생각해보세요. 즉, 하나의 정확한 단어를 찾는 대신, 원하는 텍스트의 종류를 설명하는 것입니다.

일반 검색: "John Smith 찾기" → 정확히 그 이름만 찾습니다.
정규식 검색: "전화번호처럼 보이는 모든 것 찾기" → 010-1234-5678, 02-987-6543 등과 같은 것을 찾습니다.
많은 NLP 패키지는 내부적으로 정규식을 많이 사용합니다. [redacted link]과 같은 정규식 테스터는 정규식 표현을 이해하고 만드는 데 유용한 도구입니다.

| Pattern | Meaning | Example |
|---------|---------|---------|
| `.` | any single character | `c.t` → cat, cut, c@t |
| `\d` | any digit (0–9) | `\d\d` → 42, 07 |
| `\w` | any letter/digit | `\w+` → hello, abc123 |
| `\s` | any whitespace | space, tab |
| `*` | zero or more times | `ab*` → a, ab, abbb |
| `+` | one or more times | `ab+` → ab, abbb |
| `?` | optional (0 or 1) | `colou?r` → color, colour |
| `[abc]` | any of these characters | `[aeiou]` → any vowel |
| `^` | start of line | |
| `$` | end of line | |


In [8]:
# Regex example1
import re
#//d +     //d의 차이를
text = "I have 3 cats and 12 books"
print("\\d  →", re.findall(r'\d',  text))   # each digit separately
print("\\d+ →", re.findall(r'\d+', text))   # consecutive digits as one number


\d  → ['3', '1', '2']
\d+ → ['3', '12']


In [9]:
# Regex example 2
text2 = "Hello, NLP 2026!"
print("\\w  →", re.findall(r'\w',  text2))  # every letter/digit one by one
print("\\w+ →", re.findall(r'\w+', text2))  # whole words

\w  → ['H', 'e', 'l', 'l', 'o', 'N', 'L', 'P', '2', '0', '2', '6']
\w+ → ['Hello', 'NLP', '2026']


In [10]:
# Regex example 3 공백을 하나하나 분류하든가 뭉탱뭉탱으로 하든가
text3 = "one two   three\tfour"
print("\\s  →", re.findall(r'\s',  text3))  # each space/tab
print("\\s+ →", re.findall(r'\s+', text3))  # group of spaces together


\s  → [' ', ' ', ' ', ' ', '\t']
\s+ → [' ', '   ', '\t']


In [11]:
# Regex example 4  불러오는거 틀에 맞는거
text4 = "cat cut c@t c.t coat"
print(".at  →", re.findall(r'.at',  text4))   # any char + 'at'
print("c.t  →", re.findall(r'c.t',  text4))   # c, any char, t
print("c\\.t →", re.findall(r'c\.t', text4))  # literal dot only


.at  → ['cat', 'oat']
c.t  → ['cat', 'cut', 'c@t', 'c.t']
c\.t → ['c.t']


In [12]:
# Regex example 5
nums = "1 22 333 4444 55555"
print("\\d{3}   →", re.findall(r'\d{3}',   nums))  # exactly 3 digits
print("\\d{2,4} →", re.findall(r'\d{2,4}', nums))  # 2 to 4 digits

\d{3}   → ['333', '444', '555']
\d{2,4} → ['22', '333', '4444', '5555']


In [13]:
# Regex example 6  모음 자음, 대문자 ,소문자
text5 = "apple Banana CHERRY date123"
print("[aeiou]  →", re.findall(r'[aeiou]',  text5))  # any vowel
print("[A-Z]+   →", re.findall(r'[A-Z]+',   text5))  # uppercase runs
print("[a-zA-Z]+ →", re.findall(r'[a-zA-Z]+', text5))  # only letters


[aeiou]  → ['a', 'e', 'a', 'a', 'a', 'a', 'e']
[A-Z]+   → ['B', 'CHERRY']
[a-zA-Z]+ → ['apple', 'Banana', 'CHERRY', 'date']


In [14]:
# Regex example 7 앞에 있는거 출력하려면 ^ 뒤에 있는건 달러사인
text = "hello world hello"
print("hello  →", re.findall(r'hello',  text))   # ['hello', 'hello']
print("^hello →", re.findall(r'^hello', text))   # ['hello']
print("world  →", re.findall(r'world',  text))   # ['world']
print("world$ →", re.findall(r'world$', text))   # []


hello  → ['hello', 'hello']
^hello → ['hello']
world  → ['world']
world$ → []


In [15]:
# Regex example 8
tweet = "Contact alice@gmail.com or call 010-1234-5678 #urgent"
print("emails   →", re.findall(r'\w+@\w+\.\w+',         tweet)) #이정돈 해석 할 줄 알아야지. 이메일인것 정도는 파악
print("phones   →", re.findall(r'\d{2,3}-\d{3,4}-\d{4}', tweet))
print("hashtags →", re.findall(r'#\w+',                  tweet))

emails   → ['alice@gmail.com']
phones   → ['010-1234-5678']
hashtags → ['#urgent']


Returning to our tweets dataset ...

In [20]:
# Print the second example
second_example = tweets['text'][5]
second_example

"@VirginAmerica seriously would pay $30 a flight for seats that didn't have this playing.\nit's really the only bad thing about flying VA"

In this case, we don't really want to split the tweet into a list of strings. We still expect a single string of text but would like to remove the line break completely from the string.

The string method `.strip()` effectively does the job of stripping away spaces at both ends of the text. However, it won't work in our example as the newline character is in the middle of the string.

In [ ]:
# Strip only removed blankspace at both ends
second_example.strip()

This is where regex could be really helpful.

In [18]:
import re
blankspace_pattern = r'\s+' # Write a pattern in regex
blankspace_repl = ' ' # Write a replacement string
# Replace whitespace(s) with ' '
clean_text = re.sub(pattern = blankspace_pattern,
                    repl = blankspace_repl,
                    string = second_example)
print(clean_text)

@VirginAmerica seriously would pay $30 a flight for seats that didn't have this playing. it's really the only bad thing about flying VA


Ta-da! The newline character is no longer there.

### Remove Punctuation Marks

Sometimes we are only interested in analyzing **alphanumeric characters** (i.e., the letters and numbers), in which case we might want to remove punctuation marks.

The `string` module contains a list of predefined punctuation marks. Let's print them out.

In [21]:
# Load in a predefined list of punctuation marks
from string import punctuation
print(punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In practice, to remove these punctuation characters, we can simply iterate over the text and remove characters found in the list, such as shown below in the `remove_punct` function.

In [23]:
def remove_punct(text):
    '''Remove punctuation marks in input text'''

    # Select characters not in puncutaion
    no_punct = []
    for char in text:
        if char not in punctuation:
            no_punct.append(char)

    # Join the characters into a string
    text_no_punct = ''.join(no_punct)

    return text_no_punct

Let's apply the function to the example below.

In [24]:
# Print the third example
third_example = tweets['text'][20]
print("before removal:",third_example)

# Apply the function
print("after removal:", remove_punct(third_example))


before removal: @VirginAmerica why are your first fares in May over three times more than other carriers when all seats are available to select???
after removal: VirginAmerica why are your first fares in May over three times more than other carriers when all seats are available to select


<a id='section2'></a>

# Tokenization

One of the most important steps in text analysis is tokenization. This is the process of breaking a long sequence of text into word tokens. With these tokens available, we are ready to perform word-level analysis. For instance, we can filter out tokens that don't contribute to the core meaning of the text.

In this section, we'll introduce how to perform tokenization using `nltk`, `spaCy`, and a Large Language Model (`bert`). The purpose is to expose you to different NLP packages, help you understand their functionalities, and demonstrate how to access key functions in each package.

### `nltk` 얘는 하나씩 쪼개는 느낌

The first package we'll be using is called **Natural Language Toolkit**, or `nltk`.

Let's install a couple modules from the package.

In [31]:
import nltk
# Uncomment the following lines to install these modules
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

`nltk` has a function called `word_tokenize`. It requires one argument, which is the text to be tokenized, and it returns a list of tokens for us.

In [32]:
# Load word_tokenize
from nltk.tokenize import word_tokenize

# Print the example
text = tweets['text'][7]
print(text)

@VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there. https://t.co/mWpG7grEZP


In [33]:
# Apply the NLTK tokenizer
nltk_tokens = word_tokenize(text)
nltk_tokens

['@',
 'VirginAmerica',
 'Really',
 'missed',
 'a',
 'prime',
 'opportunity',
 'for',
 'Men',
 'Without',
 'Hats',
 'parody',
 ',',
 'there',
 '.',
 'https',
 ':',
 '//t.co/mWpG7grEZP']

You may feel that accessing functions in `nltk` is pretty straightforward. The function we used above was imported from the `nltk.tokenize` module, which as the name suggests, primarily does the job of tokenization.

Underlyingly, `nltk` has [a collection of modules](https://www.nltk.org/api/nltk.html) that fulfill different purposes, to name a few:

| NLTK module   | Fucntion                  | Link                                                         |
|---------------|---------------------------|--------------------------------------------------------------|
| nltk.tokenize | Tokenization              | [Documentation](https://www.nltk.org/api/nltk.tokenize.html) |
| nltk.corpus   | Retrieve built-in corpora | [Documentation](https://www.nltk.org/nltk_data/)             |
| nltk.tag      | Part-of-speech tagging    | [Documentation](https://www.nltk.org/api/nltk.tag.html)      |
| nltk.stem     | Stemming                  | [Documentation](https://www.nltk.org/api/nltk.stem.html)     |
| ...           | ...                       | ...                                                          |

Let's import `stopwords` from the `nltk.corpus` module, which hosts a range of built-in corpora.

In [34]:
# Load predefined stop words from nltk
from nltk.corpus import stopwords

stop = set(stopwords.words('english'))
print(list(stop)[:30])

text = "The cat is sitting on the mat".split()
filtered = [w for w in text if w.lower() not in stop]

print(filtered)   # → ['cat', 'sitting', 'mat']


['shan', "should've", 'your', 'any', 'didn', 'between', 'under', 'how', "mustn't", 'doesn', 'up', 'couldn', 'm', 'their', 'again', "i'd", "weren't", "i'll", 'which', 'mightn', "shan't", 'while', 'more', 'our', 'herself', 'most', "you've", 'needn', 'you', 'during']
['cat', 'sitting', 'mat']


### `spaCy`
Other than `nltk`, we have another widely-used package called `spaCy`.

`spaCy` has its own processing pipeline. It takes in a string of text, runs the `nlp` pipeline on it, and stores the processed text and its annotations in an object called `doc`. The `nlp` pipeline always performs tokenization, as well as [other text analysis components](https://spacy.io/usage/processing-pipelines#custom-components) requested by the user. These components are pretty similar to modules in `nltk`.

Note that we always start by initializing the `nlp` pipeline, depending on the language of the text. Here, we are loading a pretrained language model for English: `en_core_web_sm`. The name suggests that it is a lightweight model trained on some text data (e.g., blogs); see model descriptions [here](https://spacy.io/models/en#en_core_web_sm).

This is the first time we encounter the concept of **pretraining**, though you may have heard it elsewhere. In the context of NLP, pretraining means that the model has been trained on a vast amount of data. As a result, it comes with a certain "knowledge" of word structure and grammar of the language.

Therefore, when we apply the model to our own data, we can expect it to be reasonably accurate in performing various annotation tasks, e.g., tagging a word's part of speech, identifying the syntactic head of a phrase, and etc.

Let's dive in! We'll first need to load the pretrained language model we installed earlier.

In [35]:
import spacy
nlp = spacy.load('en_core_web_sm')

The `nlp` pipeline, by default, includes a set of components, which we can access via the `.pipe_names` attribute.

You may notice that it dosen't include the tokenizer. Don't worry! Tokenizer is a special component that the pipeline always includes.

In [37]:
# Retrieve components included in NLP pipeline
nlp.pipe_names

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

Let's run the `nlp` pipeline on our example tweet data, and assign it to a variable `doc`.

In [39]:
# Apply the pipeline to example tweet
doc = nlp(tweets['text'][7])

Under the hood, the `doc` object contains the tokens (created by the tokenizer) and their annotations (created by other components), which are [linguistic features](
https://spacy.io/usage/linguistic-features) useful for text analysis. We retrieve the token and its annotations by accessing corresponding attributes.

| Attribute      | Annotation                              | Link                                                                      |
|----------------|-----------------------------------------|---------------------------------------------------------------------------|
| token.text     | The token in verbatim text              | [Documentation](https://spacy.io/api/token#attributes)                    |
| token.is_stop  | Whether the token is a stop word        | [Documentation](https://spacy.io/api/attributes#_title)                   |
| token.is_punct | Whether the token is a punctuation mark | [Documentation](https://spacy.io/api/attributes#_title)                   |
| token.lemma_   | The base form of the token              | [Documentation](https://spacy.io/usage/linguistic-features#lemmatization) |
| token.pos_     | The simple POS-tag of the token         | [Documentation](https://spacy.io/usage/linguistic-features#pos-tagging)   |
| ...            | ...                                     | ...                                                                       |

Let's first get the tokens themselves! We'll iterate over the `doc` object and retrieve the text of each token.

In [40]:
# Get the verbatim texts of tokens
spacy_tokens = [token.text for token in doc]
spacy_tokens

['@VirginAmerica',
 'Really',
 'missed',
 'a',
 'prime',
 'opportunity',
 'for',
 'Men',
 'Without',
 'Hats',
 'parody',
 ',',
 'there',
 '.',
 'https://t.co/mWpG7grEZP']

In [41]:
# cf. NLTK tokens
nltk_tokens

['@',
 'VirginAmerica',
 'Really',
 'missed',
 'a',
 'prime',
 'opportunity',
 'for',
 'Men',
 'Without',
 'Hats',
 'parody',
 ',',
 'there',
 '.',
 'https',
 ':',
 '//t.co/mWpG7grEZP']

We can also access various annotations of these okens. For instance, one annotation `spaCy` offers is that it conveniently encodes whether a token is a stop word.

In [42]:
# Retrieve the is_stop annotation  #차이점
spacy_stops = [token.is_stop for token in doc]

# The results are boolean values
spacy_stops

[False,
 True,
 False,
 True,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 True,
 False,
 False]

### Cool Feature of spaCy: Named Entity Recognition
(NER)

One of the cool features of spaCy is its ability to automatically recognize named entities in text — names of people, places, organizations, dates, money amounts, and more. Instead of manually searching for these patterns with regex, spaCy uses a pretrained model that understands context.

For example, in the sentence "Apple opened a new store in Seoul last Monday", spaCy can identify:
- "Apple" as an organization (ORG)
- "Seoul" as a geopolitical entity (GPE)
- "last Monday" as a date (DATE)

개체명 인식(NER)이란?
NER은 텍스트에서 사람, 장소, 조직, 날짜, 금액 등과 같이 '고유한 이름'을 가진 단어나 구(개체명)를 자동으로 찾아내고, 해당 개체명이 어떤 유형에 속하는지 분류하는 기술입니다.

예를 들어, "Apple이 서울에 지난 월요일 새 매장을 열었다"는 문장에서 spaCy는 다음과 같이 개체명을 식별합니다:
### Entity types spaCy recognizes

Out of the box, spaCy's English model can identify 18 entity categories:

| Label | Meaning | Example |
|-------|---------|---------|
| `PERSON` | People, including fictional | Barack Obama, Harry Potter |
| `NORP` | Nationalities, religious or political groups | American, Buddhist, Democrats |
| `FAC` | Buildings, airports, highways, bridges | the Eiffel Tower, JFK Airport |
| `ORG` | Companies, agencies, institutions | Google, the UN, Harvard |
| `GPE` | Countries, cities, states | Korea, Seoul, California |
| `LOC` | Non-GPE locations (mountains, oceans) | the Pacific Ocean, Mt. Everest |
| `PRODUCT` | Objects, vehicles, foods | iPhone, Boeing 747 |
| `EVENT` | Named hurricanes, wars, sports events | World War II, the Olympics |
| `WORK_OF_ART` | Titles of books, songs, films | Hamlet, Bohemian Rhapsody |
| `LAW` | Named documents made into laws | Roe v. Wade |
| `LANGUAGE` | Any named language | Korean, Spanish |
| `DATE` | Absolute or relative dates | 2026-05-07, last Monday |
| `TIME` | Times smaller than a day | 3:30 PM, midnight |
| `PERCENT` | Percentages | 50%, ninety percent |
| `MONEY` | Monetary values | $100, 5만원 |
| `QUANTITY` | Measurements | 10 km, 5 kilograms |
| `ORDINAL` | "first", "second" | first, 3rd |
| `CARDINAL` | Numerals not covered by other types | one, 42 |

You can always check the meaning of any label using `spacy.explain()`:

```python
import spacy
spacy.explain("GPE")   # → 'Countries, cities, states'
spacy.explain("ORG")   # → 'Companies, agencies, institutions, etc.'

spaCy가 인식하는 개체 유형
spaCy의 영어 모델은 기본적으로 18가지의 다양한 개체 범주를 인식할 수 있습니다. 몇 가지 주요 예시는 다음과 같습니다:

PERSON: 사람의 이름 (예: Barack Obama, Harry Potter)
ORG: 회사, 기관, 단체 (예: Google, UN, Harvard)
GPE: 국가, 도시, 주 (예: Korea, Seoul, California)
LOC: 지리적 위치 (예: Pacific Ocean, Mt. Everest)
DATE: 날짜 (예: 2026-05-07, last Monday)
MONEY: 금액 (예: $100, 5만원)



In [43]:
# GPE example
tweet_city = tweets['text'][8273]
print(tweet_city)

# Visualize the identified entities
from spacy import displacy
doc_city = nlp(tweet_city)
displacy.render(doc_city, style='ent', jupyter=True)

@JetBlue Vegas, San Francisco, Baltimore, San Diego and Philadelphia so far! I'm a very frequent business traveler.


In [45]:
# But use with caution!!!
tweet_airport = tweets['text'][502]
print(tweet_airport)
doc_airport = nlp(tweet_airport)

# Visualize the identified entities
displacy.render(doc_airport, style='ent', jupyter=True)

@VirginAmerica Flying LAX to SFO and after looking at the awesome movie lineup I actually wish I was on a long haul.


## Tokenizers Since LLMs

So far, we've seen what tokenization looks like with two widely-used NLP packages. They work quite well in some settings, but not others. Recall that `nltk` struggles with URLs.

Now, imagine the data we have is even messier, containing misspellings, recently coined words, foreign names, and etc (collectively called "out of vocabulary" or OOV words). In such circumstances, we might need a more powerful model to handle these complexities.

In fact, tokenization schemes change substantially with **Large Language Models** (LLMs), which are models trained on an enormous amount of data from mixed sources. With that magnitude of data, LLMs are better at chunking a longer sequence into tokens and tokens into **subtokens**. These subtokens can be morphological units of a word, such as an affix, but they can also be parts of a word where the model sets a "meaningful" boundary.

In this section, we will demonstrate tokenization in **BERT** (Bidirectional Encoder Representations from Transformers), which utilizes a tokenization algorithm called [**WordPiece**](https://huggingface.co/learn/nlp-course/en/chapter6/6).

We will load the tokenizer of BERT from the package `transformers`, which hosts a number of Transformer-based LLMs (e.g., BERT). We won't go into the architecture of Transformer in this tutorial, but feel free to check out the D-lab workshop on [GPT Fundamentals](https://github.com/dlab-berkeley/GPT-Fundamentals)!

### WordPiece Tokenization

Note that BERT comes in a variety of versions. The one we will explore today is `bert-base-uncased`. This model has a moderate size (referred to as `base`) and is case-insensitive, meaning the input text will be lowercased by default.

In [46]:
# Load BERT tokenizer in
from transformers import BertTokenizer

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The tokenizer has multiple functions, as we will see in a minute. Now we want to access the `.tokenize()` function from the tokenizer.

Let's tokenize an example tweet below. What have you noticed? The double "hashtag" symbols (`##`) refer to a subword token—a segment separated from the previous token.

In [47]:
# Select an example tweet from dataframe
text = tweets['text'][194]

# Apply tokenizer
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

Tokens: ['@', 'virgin', '##ame', '##rica', 'just', 'd', '##m', "'", 'd', '.', 'same', 'issue', 'persist', '##ing', '.']
Number of tokens: 15



One significant development with LLMs is that each token is assigned an ID from its vocabulary. Our computer does not understand text in its raw form, so each token is translated into an ID. These IDs are the inputs that the model accesses and operates on.

Tokens and IDs can be converted bidirectionally, for example:

In [48]:
# Get the input ID of the word
print(f"ID of just is: {tokenizer.vocab['just']}")

# Get the text of the input ID
print(f"Token 2074 is: {tokenizer.decode([2074])}")

ID of just is: 2074
Token 2074 is: just


Let's convert tokens to input IDs.

In [49]:
# Convert a list of tokens to a list of input IDs
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Number of input IDs: {len(input_ids)}")
print(f"Input IDs of text: {input_ids}")

Number of input IDs: 15
Input IDs of text: [1030, 6261, 14074, 14735, 2074, 1040, 2213, 1005, 1040, 1012, 2168, 3277, 29486, 2075, 1012]


### Special Tokens

In addition to the tokens and subtokens discussed above, BERT also makes use of three special tokens: `SEP`, `CLS`, and `UNK`. The `SEP` token acts as a sentence terminator, commonly known as an `EOS` (End of Sentence) token. The `UNK` token represents any token that is not found in the vocabulary, hence "unknown" tokens. The `CLS` token is added to the beginning of the sentence. It originates from text classification tasks (e.g., spam detection), where reseachers found it useful to have a token that aggregates the information of the entire sentence for classification purposes.

When we apply `tokenizer()` directly to our text data, we are asking BERT to **encode** the text for us. This involves multiple steps:
- Tokenize the text
- Add special tokens
- Convert tokens to input IDs
- Other model-specific processes
  
Let's print them out.

In [50]:
# Get the input IDs by providing the key
input_ids_from_tokenizer = tokenizer(text)['input_ids']
print(f"Number of input IDs: {len(input_ids_from_tokenizer)}")
print(f"IDs from tokenizer: {input_ids_from_tokenizer}")

Number of input IDs: 17
IDs from tokenizer: [101, 1030, 6261, 14074, 14735, 2074, 1040, 2213, 1005, 1040, 1012, 2168, 3277, 29486, 2075, 1012, 102]


It looks like we have two more tokens added: 101 and 102.

Let's convert them to texts!

In [ ]:
# Convert input IDs to texts
print(f"The 101st token: {tokenizer.convert_ids_to_tokens(101)}")
print(f"The 102nd token: {tokenizer.convert_ids_to_tokens(102)}")

As you can see, our text example is now a list of vocabulary IDs. In addtion to that, BERT adds the sentence terminator `SEP` and the beginning `CLS` token to the original text. BERT's tokenizer encodes tons of texts likewise; and afterwards, they are ready for further processes.

<div class="alert alert-success">

## ❗ Key Points

* Preprocessing includes multiple steps, some of them are more common to text data regardlessly, and some are task-specific.
* Both `nltk` and `spaCy` could be used for tokenization and stop word removal. The latter is more powerful in providing various linguistic annotations.
* Tokenization works differently in BERT, which often involves breaking down a whole word into subwords.

</div>

### More advanced tutorials:

1. A tutorial introducing the tokenization scheme in BERT: [The huggingface NLP course on wordpiece tokenization](https://huggingface.co/learn/nlp-course/chapter6/6?fw=pt)
2. A specific example of "failure" in tokenization: [Weaknesses of wordpiece tokenization: Findings from the front lines of NLP at VMware.](https://medium.com/@rickbattle/weaknesses-of-wordpiece-tokenization-eb20e37fec99)
3. How does BERT decide boundaries between subtokens: [Subword tokenization in BERT](https://tinkerd.net/blog/machine-learning/bert-tokenization/#subword-tokenization)

# 한국어 텍스트 전처리 (Korean Text Preprocessing)

지금까지 살펴본 영어 전처리 도구들(`nltk`, `spaCy`, `BERT`)은 한국어에 그대로 적용하기 어려움. 한국어는 영어와 언어학적 특성이 매우 다르기 때문.

### 한국어가 까다로운 이유

1. 교착어(agglutinative): 한국어는 어간에 조사·어미가 계속 붙는 구조.
   예) "학교" + "에서" + "는" → "학교에서는"  
   → 공백 기준 단어 분리(`split()`)만으로는 의미 단위를 얻을 수 없음.

2. 형태소 분석(morphological analysis)이 필수: "갔습니다"를 "가 + 았 + 습니다"로 분해해야 동사 "가다"를 인식할 수 있음.

3. 띄어쓰기가 불안정: 같은 문장도 사람마다 다르게 띄어 씀.  
   예) "잘부탁드립니다" / "잘 부탁드립니다" / "잘부탁 드립니다"

4. 풍부한 어미 변화: "먹다, 먹고, 먹어서, 먹으면, 먹었다, 먹겠다…" — 어미마다 다른 토큰으로 처리되면 같은 동사임을 놓침.

이런 이유로 한국어 NLP에서는 단어 토큰화 대신 형태소 토큰화를 표준으로 쓴다.


In [ ]:
# Uncomment to install
# !pip install konlpy
# !pip install kiwipiepy
# KoNLPy requires Java (JDK 8+). On macOS: brew install openjdk

### KoNLPy — Okt 분석기

KoNLPy는 여러 형태소 분석기를 한 번에 제공합니다. 그중 Okt(Open Korean Text)는 트위터 데이터를 위해 개발되어 SNS 텍스트에 비교적 잘 맞음.

| 메서드 | 기능 |
|--------|------|
| `.morphs(text)` | 형태소 단위로 분리 |
| `.nouns(text)` | 명사만 추출 |
| `.pos(text)` | 형태소 + 품사 태그 |
| `.normalize(text)` | 구어체 정규화 (Okt 전용) |

한국어 분석기의 종류 및 비교:
https://konlpy.org/ko/latest/morph/


In [ ]:
from konlpy.tag import Okt

okt = Okt()

# 항공사 트윗 스타일의 한국어 예시
sample = "@대한항공 어제 인천공항에서 진짜 오래 기다렸어요ㅠㅠ 다음엔 안 탈래요 #불만"

print("형태소:", okt.morphs(sample))
print("명사:", okt.nouns(sample))
print("품사:", okt.pos(sample))
print("정규화:", okt.normalize(sample))

In [ ]:
# 분석기 비교
from konlpy.tag import Kkma, Komoran

kkma = Kkma()
komoran = Komoran()

text = "아버지가방에들어가신다"

print("Okt:    ", okt.morphs(text))
print("Kkma:   ", kkma.morphs(text))
print("Komoran:", komoran.morphs(text))


### Kiwi: 최신 형태소 분석기

Kiwi(`kiwipiepy`)는 최근 한국어 NLP에서 가장 주목받는 형태소 분석기. C++로 구현되어 KoNLPy보다 훨씬 빠르고, 신조어·구어체에 강함. Java 설치도 필요 없음.

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 항공사 트윗 스타일의 한국어 예시
sample = "@대한항공 어제 인천공항에서 진짜 오래 기다렸어요ㅠㅠ 다음엔 안 탈래요 #불만"

# 토큰화
result = kiwi.tokenize(sample)
for token in result:
    print(f"{token.form:10s} | {token.tag:5s} | start={token.start}, len={token.len}")


In [ ]:
# 문장 분리 예시
paragraph = "비행기가 연착됐어요. 환불은 어디서 받죠? 고객센터 전화도 안 받네요."
for sent in kiwi.split_into_sents(paragraph):
    print("→", sent.text)

### 한국어 불용어 처리

영어와 달리 한국어는 표준화된 불용어 사전이 없어 직접 만들거나 공개된 리스트(예: [한국어 stopwords 모음](https://www.ranks.nl/stopwords/korean))를 활용함. 보통 조사·어미·일부 부사를 제거.

In [ ]:
korean_stopwords = {
    '을', '를', '이', '가', '은', '는', '의', '에', '에서', '와', '과',
    '도', '만', '으로', '로', '에게', '한테', '부터', '까지',
    '있다', '하다', '되다', '이다', '같다',
    '그', '저', '이런', '저런', '그런'
}

tokens = okt.morphs(sample)
filtered = [t for t in tokens if t not in korean_stopwords and len(t) > 1]
print("원본:", tokens)
print("필터:", filtered)

# 품사 기반 필터링 방법
# 명사·동사·형용사만 남기기
pos_tagged = okt.pos(sample, stem=True)  # stem=True → 어간 추출
content_words = [w for w, p in pos_tagged if p in ('Noun', 'Verb', 'Adjective')]
print(content_words)

### LLM 시대의 한국어 토크나이저 — KLUE-BERT

영어에서 BERT가 WordPiece로 서브워드를 나누듯, 한국어에도 LLM 기반 토크나이저가 있음. 대표적으로 KLUE 프로젝트의 `klue/bert-base`는 한국어 코퍼스로 학습된 BERT 모델.

영어 BERT의 `##` 표기처럼 한국어 토크나이저도 단어 중간에서 시작되는 서브워드를 표시. 형태소 분석기와 달리 사전에 없는 신조어·외래어도 서브워드로 쪼개서 처리할 수 있음.

In [ ]:

from transformers import AutoTokenizer

ko_tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

text = "대한항공 고객센터에 전화했는데 너무 오래 기다렸어요"

tokens = ko_tokenizer.tokenize(text)
print(f"토큰 수: {len(tokens)}")
print(f"토큰: {tokens}")

# 형태소 분석기와 비교
print(f"\nOkt:  {okt.morphs(text)}")
print(f"Kiwi: {[t.form for t in kiwi.tokenize(text)]}")


In [ ]:
# 신조어/외래어 처리 비교
oov_text = "에어비앤비에서 노쇼당했어요 ㅋㅋㅋ"

print("KLUE-BERT:", ko_tokenizer.tokenize(oov_text))
print("Okt:      ", okt.morphs(oov_text))
print("Kiwi:     ", [t.form for t in kiwi.tokenize(oov_text)])
